# AgriCL-Net, step 5: package the trained models for the app

Collects everything the Streamlit app needs into one folder, `MyDrive/Colab Notebooks/agricl-net/app_bundle`, tests it on
a few photos (on GPU, and on CPU to show the speed on a laptop) and zips it for download.

**Needs:** a finished run of notebook 01 (lesion model), plus notebook 03 (joint classifier) or
notebook 04 (continual-learning classifier).

## Settings

In [ ]:
SIZE = 'l'                    # lesion run settings from notebook 01
IMG_SIZE = 352
MODEL = 'swin_tiny_patch4_window7_224.ms_in22k_ft_in1k'
CLASSIFIER = 'continual'      # 'continual' (notebook 04, replay_kd) or 'joint' (notebook 03)
LEAF = 'sam2'                 # default leaf method in the app: 'sam2' or 'exg'

## Storage and project code from GitHub

Works on **Colab** and **Kaggle**:
* Colab: everything is kept in Google Drive, `MyDrive/Colab Notebooks/agricl-net`.
* Kaggle: turn on *Settings > Accelerator > GPU T4* and *Settings > Internet*. Results go to
  `/kaggle/working/agricl-net` and are kept when you *Save Version*. Downloads go to `/tmp` (fetched again each
  session). To use an earlier notebook's results (or resume a run), attach that notebook's output with
  *Add Input > Notebook Output*: it is copied in automatically below.

In [ ]:
import os, sys, glob, shutil
ON_KAGGLE = os.path.exists('/kaggle/working')
if ON_KAGGLE:
    WORK = '/tmp'                               # scratch space: code, unzipped data
    DRIVE = '/kaggle/working/agricl-net'        # results, saved with each notebook version
    CACHE = '/tmp/agricl-cache'                 # dataset zip and SAM2 checkpoint
    for src in glob.glob('/kaggle/input/**/agricl-net/runs/*', recursive=True):
        dst = f'{DRIVE}/runs/{os.path.basename(src)}'
        if os.path.isdir(src) and not os.path.exists(dst):
            shutil.copytree(src, dst)
            print('copied earlier results:', os.path.basename(src))
else:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content'
    DRIVE = '/content/drive/MyDrive/Colab Notebooks/agricl-net'   # dataset zip, SAM2 checkpoint and all results
    CACHE = f'{DRIVE}/cache'
os.makedirs(CACHE, exist_ok=True)
os.makedirs(f'{DRIVE}/runs', exist_ok=True)

CODE = f'{WORK}/AGRICL-NET'
if os.path.exists(CODE):
    !git -C {CODE} pull -q
else:
    !git clone -q https://github.com/Dev02005/AGRICL-NET.git {CODE}
sys.path.insert(0, CODE)
!git -C {CODE} log -1 --format='code version: %h %s'

## SAM2-UNet code and SAM2 checkpoint

In [ ]:
!pip -q install hydra-core==1.3.2
SAM2UNET = f'{WORK}/SAM2-UNet'
if not os.path.exists(SAM2UNET):
    !git clone -q https://github.com/WZH0120/SAM2-UNet.git {SAM2UNET}
    !git -C {SAM2UNET} checkout -q 01598e5e9912ffb23f965ecbebf4d1dfecbaa56e

from agricl.models.sam2unet import CHECKPOINTS, CHECKPOINT_URL
CKPT = f'{CACHE}/{CHECKPOINTS[SIZE]}'
if not os.path.exists(CKPT):
    !wget -q --show-progress -O "{CKPT}" "{CHECKPOINT_URL}{CHECKPOINTS[SIZE]}"
print(CKPT, os.path.getsize(CKPT) // 2**20, 'MB')

## Build the bundle

In [ ]:
import json, shutil
!pip -q install timm

BUNDLE = f'{DRIVE}/app_bundle'
os.makedirs(BUNDLE, exist_ok=True)
short = MODEL.split('.')[0]

lesion_run = f"{DRIVE}/runs/lesion_sam2unet_{SIZE.replace('+', 'p')}_{IMG_SIZE}"
assert os.path.exists(f'{lesion_run}/best.pt'), 'run notebook 01 (SMOKE_TEST = False) first'
shutil.copy(f'{lesion_run}/best.pt', f'{BUNDLE}/lesion_best.pt')
shutil.copy(CKPT, f'{BUNDLE}/{os.path.basename(CKPT)}')
shutil.copytree(SAM2UNET, f'{BUNDLE}/SAM2-UNet', dirs_exist_ok=True,
                ignore=shutil.ignore_patterns('.git', '*.jpg', '*.drawio'))

if CLASSIFIER == 'joint':
    run = f'{DRIVE}/runs/classifier_{short}'
    assert os.path.exists(f'{run}/best.pt'), 'run notebook 03 (SMOKE_TEST = False) first'
    shutil.copy(f'{run}/best.pt', f'{BUNDLE}/classifier.pt')
    shutil.copy(f'{run}/classes.json', f'{BUNDLE}/classes.json')
    shutil.copy(f'{run}/calibration.json', f'{BUNDLE}/calibration.json')
    kind = 'timm'
else:
    run = f'{DRIVE}/runs/continual_{short}'
    assert os.path.exists(f'{run}/replay_kd_final.pt'), 'run notebook 04 (SMOKE_TEST = False) first'
    shutil.copy(f'{run}/replay_kd_final.pt', f'{BUNDLE}/classifier.pt')
    rounds = json.load(open(f'{run}/rounds.json'))
    json.dump([d for r in rounds for d in r['diseases']], open(f'{BUNDLE}/classes.json', 'w'), indent=1)
    cal = json.load(open(f'{run}/results.json'))['replay_kd']['calibration']
    json.dump({'temperature': cal['temperature'], 'unknown_threshold': cal['unknown_threshold']},
              open(f'{BUNDLE}/calibration.json', 'w'), indent=2)
    kind = 'incremental'

json.dump({
    'lesion': {'size': SIZE, 'img_size': IMG_SIZE, 'weights': 'lesion_best.pt',
               'sam2_checkpoint': os.path.basename(CKPT), 'sam2unet_repo': 'SAM2-UNet'},
    'classifier': {'kind': kind, 'model_name': MODEL, 'img_size': 224, 'weights': 'classifier.pt',
                   'classes': 'classes.json', 'calibration': 'calibration.json'},
    'leaf': {'method': LEAF},
}, open(f'{BUNDLE}/agricl_bundle.json', 'w'), indent=2)
!ls -la "{BUNDLE}"

## Test the bundle on unseen-disease and seen-disease photos (GPU, then CPU)

In [ ]:
import time, random
from PIL import Image
from agricl.predictor import AgriCLPredictor

DATA = f'{WORK}/data/plantseg'
if not os.path.exists(f'{DATA}/Metadata.csv'):
    if not os.path.exists(f'{CACHE}/plantseg.zip'):
        !wget -q --show-progress -O "{CACHE}/plantseg.zip" "https://zenodo.org/records/17719108/files/plantseg.zip?download=1"
    !unzip -q "{CACHE}/plantseg.zip" -d "{WORK}/data"
from agricl.data.plantseg import load_records, split_seen_unseen
seen, unseen = split_seen_unseen(load_records(DATA))
samples = random.Random(0).sample(unseen, 2) + random.Random(0).sample([r for r in seen if r.split == 'test'], 2)

for device in ('cuda', 'cpu'):
    p = AgriCLPredictor(BUNDLE, device=device)
    p.diagnose(Image.open(samples[0].image))  # warm-up
    for r in samples:
        t = time.time()
        d = p.diagnose(Image.open(r.image))
        print(f"[{device}] true: {r.disease:<28} -> {d.decision.status:<10} {str(d.decision.disease):<28} "
              f"severity {d.severity.percent:5.1f}% {d.severity.level:<8} {time.time() - t:5.1f} s")
    del p

## Zip for download

In [ ]:
shutil.make_archive(f'{DRIVE}/agricl_app_bundle', 'zip', BUNDLE)
print(f"{DRIVE}/agricl_app_bundle.zip", os.path.getsize(f'{DRIVE}/agricl_app_bundle.zip') // 2**20, 'MB')

## Next
Download `MyDrive/Colab Notebooks/agricl-net/agricl_app_bundle.zip`, unzip it to `AGRI-NET/models/bundle/` on your computer and
start the app (see the README, section "Run the app"). The CPU times above are roughly what a laptop will need
per photo; choose the colour-index leaf method in the app's sidebar to make it faster.